# Telco Customer Churn Prediction


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pickle, io
from google.colab import files

uploaded = files.upload()
nama_file = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[nama_file]))
print(df.shape)


In [ ]:
df['TotalCharges']=pd.to_numeric(df['TotalCharges'],errors='coerce')
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
df['target_churn']=df['Churn'].apply(lambda x:1 if x=='Yes' else 0)
le_internet=LabelEncoder()
df['internet_encoded']=le_internet.fit_transform(df['InternetService'])
le_contract=LabelEncoder()
df['contract_encoded']=le_contract.fit_transform(df['Contract'])
Q1=df['MonthlyCharges'].quantile(0.25)
Q3=df['MonthlyCharges'].quantile(0.75)
IQR=Q3-Q1
lower_bound=Q1-1.5*IQR
upper_bound=Q3+1.5*IQR
df_clean=df[(df['MonthlyCharges']>=lower_bound)&(df['MonthlyCharges']<=upper_bound)].copy()


In [ ]:
plt.figure(figsize=(18,5))
plt.subplot(1,3,1)
sns.boxplot(data=df_clean,x='MonthlyCharges')
plt.subplot(1,3,2)
sns.countplot(data=df_clean,x='target_churn')
plt.subplot(1,3,3)
fitur=df_clean[['tenure','MonthlyCharges','internet_encoded','contract_encoded','target_churn']]
sns.heatmap(fitur.corr(),annot=True,cmap='coolwarm',fmt='.2f')
plt.tight_layout(); plt.show()


In [ ]:
X=df_clean[['tenure','MonthlyCharges','internet_encoded','contract_encoded']]
y=df_clean['target_churn']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)
model_dt=DecisionTreeClassifier(random_state=42,max_depth=6)
model_dt.fit(X_train_scaled,y_train)
model_rf=RandomForestClassifier(random_state=42,n_estimators=100,max_depth=6)
model_rf.fit(X_train_scaled,y_train)


In [ ]:
def tampilkan_evaluasi(model,X_test,y_test,nama):
 y_pred=model.predict(X_test)
 print('===',nama,'===')
 print('Accuracy',round(accuracy_score(y_test,y_pred)*100,2))
 print('Precision',round(precision_score(y_test,y_pred)*100,2))
 print('Recall',round(recall_score(y_test,y_pred)*100,2))
 print('F1',round(f1_score(y_test,y_pred)*100,2))
 cm=confusion_matrix(y_test,y_pred)
 sns.heatmap(cm,annot=True,fmt='d',cmap='Reds'); plt.show()
tampilkan_evaluasi(model_dt,X_test_scaled,y_test,'Decision Tree')
tampilkan_evaluasi(model_rf,X_test_scaled,y_test,'Random Forest')


In [ ]:
with open('rf_telco_model.pkl','wb') as f: pickle.dump(model_rf,f)
with open('scaler_telco.pkl','wb') as f: pickle.dump(scaler,f)
with open('le_internet.pkl','wb') as f: pickle.dump(le_internet,f)
with open('le_contract.pkl','wb') as f: pickle.dump(le_contract,f)
